In [4]:
import os
import numpy as np
import laspy
import glob
import heapq
from tqdm import tqdm
from scipy.spatial import KDTree
from sklearn.cluster import DBSCAN


In [5]:

# ==========================================
# KONFIGURACJA ŚCIEŻEK
# ==========================================
ORIGINAL_SCENE_DIR = r"C:\Users\lukas\OneDrive\Dokumenty\GeoAI_MODULES\GeoAI_podyplomowe\4_podstawy_sztucznej_inteligencji\data\test_scenes"
RAW_PREDICTIONS_DIR = r"C:\Users\lukas\OneDrive\Dokumenty\GeoAI_MODULES\GeoAI_podyplomowe\4_podstawy_sztucznej_inteligencji\data\pred\npz"
OUTPUT_DIR          = r"C:\Users\lukas\OneDrive\Dokumenty\GeoAI_MODULES\GeoAI_podyplomowe\4_podstawy_sztucznej_inteligencji\data\pred\merged"

In [6]:
# PARAMETRY SEMANTYCZNE
NUM_CLASSES = 3      # 0: background, 1: trunk, 2: crown
FILTER_CLASS = 3     # Klasa do usunięcia (out-points)

# PARAMETRY INSTANCYJNE (DBSCAN)
DBSCAN_EPS = 0.8
DBSCAN_MIN_SAMPLES = 15

# PARAMETRY WATERSHED (ROZROST KORONY)
WATERSHED_RADIUS = 1.0       # Promień sąsiedztwa
MIN_STEM_HEIGHT = 1.5        # Minimalna wysokość pnia by uznać go za drzewo
MIN_CANOPY_POINTS = 100      # Ile punktów korony musi być nad pniem

# ==========================================
# FUNKCJE POMOCNICZE
# ==========================================

def propagate_labels(points, sparse_labels, radius=0.3):
    """Propaguje etykiety z rzadkiej chmury na gęstą chmurę oryginalną."""
    dense_labels = sparse_labels.copy()
    unlabeled_mask = (dense_labels == 0)
    
    if not np.any(unlabeled_mask): return dense_labels

    for class_id in np.unique(sparse_labels[sparse_labels > 0]):
        source_mask = (sparse_labels == class_id)
        if not np.any(source_mask): continue
        
        tree = KDTree(points[source_mask])
        dist, _ = tree.query(points[unlabeled_mask], k=1)
        
        claim_mask = dist < radius
        indices = np.where(unlabeled_mask)[0][claim_mask]
        dense_labels[indices] = class_id
        unlabeled_mask[indices] = False
        
    return dense_labels

def cluster_stems(points, semantic_labels):
    """Grupowanie pni za pomocą DBSCAN."""
    stem_mask = (semantic_labels == 1) # Klasa Trunk
    if not np.any(stem_mask): return np.zeros(len(points), dtype=int)
    
    db = DBSCAN(eps=DBSCAN_EPS, min_samples=DBSCAN_MIN_SAMPLES).fit(points[stem_mask])
    instance_labels = np.zeros(len(points), dtype=np.int32)
    instance_labels[stem_mask] = db.labels_ + 1
    return instance_labels

def grow_crowns_watershed(points, semantic_labels, stem_instances):
    """Rozrost instancji z pni na korony (Watershed z heurystyką Pillar)."""
    unique_stems = np.unique(stem_instances[stem_instances > 0])
    crown_mask = (semantic_labels == 2)
    crown_points = points[crown_mask]
    
    if len(unique_stems) == 0 or len(crown_points) == 0:
        return stem_instances

    # Przygotowanie nasion (Super-Seeds) z pni
    seed_points = points[stem_instances > 0]
    seed_labels = stem_instances[stem_instances > 0]
    
    # Inicjalizacja kolejki priorytetowej
    seed_tree = KDTree(seed_points)
    dist, closest_idx = seed_tree.query(crown_points, k=1)
    
    pq = []
    for i in range(len(crown_points)):
        heapq.heappush(pq, (dist[i], i, seed_labels[closest_idx[i]]))
    
    # Flooding
    crown_tree = KDTree(crown_points)
    visited = np.zeros(len(crown_points), dtype=bool)
    crown_inst_labels = np.zeros(len(crown_points), dtype=np.int32)
    
    while pq:
        cost, idx, inst_id = heapq.heappop(pq)
        if visited[idx]: continue
        visited[idx] = True
        crown_inst_labels[idx] = inst_id
        
        neighbors = crown_tree.query_ball_point(crown_points[idx], r=WATERSHED_RADIUS)
        for n in neighbors:
            if not visited[n]:
                new_cost = np.linalg.norm(crown_points[idx] - crown_points[n])
                heapq.heappush(pq, (cost + new_cost, n, inst_id))
    
    final_labels = stem_instances.copy()
    final_labels[crown_mask] = crown_inst_labels
    return final_labels

# ==========================================
# GŁÓWNY PIPELINE
# ==========================================

def process_all():
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    scenes = glob.glob(os.path.join(ORIGINAL_SCENE_DIR, "*.la*"))
    
    for scene_path in scenes:
        name = os.path.splitext(os.path.basename(scene_path))[0]
        print(f"\n>>> PRZETWARZANIE SCENY: {name}")
        
        # 1. Stitching NPZ -> Original Size
        las = laspy.read(scene_path)
        pts = np.vstack((las.x, las.y, las.z)).T
        num_pts = len(pts)
        
        logits_sum = np.zeros((num_pts, NUM_CLASSES))
        counts = np.zeros(num_pts)
        
        patches = glob.glob(os.path.join(RAW_PREDICTIONS_DIR, f"{name}*.npz"))
        for p_path in tqdm(patches, desc="  Stitching patches"):
            data = np.load(p_path)
            idx, logits = data['global_idx'], data['semantic_logits']
            np.add.at(logits_sum, idx, logits)
            np.add.at(counts, idx, 1)
            
        # Średnia i etykiety
        mask = counts > 0
        final_sem = np.zeros(num_pts, dtype=int)
        final_sem[mask] = np.argmax(logits_sum[mask] / counts[mask, None], axis=1)
        
        # 2. Propagacja (Full Cloud)
        final_sem = propagate_labels(pts, final_sem)
        
        # 3. Filtracja (usuwanie klasy 3)
        keep = (las.classification != FILTER_CLASS)
        pts_f = pts[keep]
        sem_f = final_sem[keep]
        las_f = las.points[keep]
        
        # 4. Segmentacja Instancyjna
        stems = cluster_stems(pts_f, sem_f)
        instances = grow_crowns_watershed(pts_f, sem_f, stems)
        
        # 5. Zapis
        out_las = laspy.LasData(las.header)
        out_las.points = las_f
        out_las.add_extra_dim(laspy.ExtraBytesParams(name="prediction", type=np.int32))
        out_las.add_extra_dim(laspy.ExtraBytesParams(name="instance_id", type=np.int32))
        out_las.prediction = sem_f
        out_las.instance_id = instances
        
        out_path = os.path.join(OUTPUT_DIR, f"{name}_final.laz")
        out_las.write(out_path)
        print(f"  Sukces! Zapisano: {out_path}")

if __name__ == "__main__":
    process_all()


>>> PRZETWARZANIE SCENY: NIBIO_plot_12_annotated


  Stitching patches:   0%|          | 0/4 [00:00<?, ?it/s]


KeyError: 'global_idx is not a file in the archive'